In [1]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import webview
import os
import glob

import zarr

In [23]:
def filter_minflux_data(file_path, verbose=True):
    """Load and filter MINFLUX data from a single file."""
    MFX_Data = np.load(file_path)
    if verbose:
        print(f'\n--- Processing: {os.path.basename(file_path)} ---')
        print('all raw                  ', len(MFX_Data))

    # Keep only valid localizations
    MFX_Data = MFX_Data[MFX_Data['vld'] == True]
    if verbose:
        print('only valid_              ', len(MFX_Data))

    # Keep only last iteration
    MFX_Data = MFX_Data[MFX_Data['itr'] == np.max(MFX_Data['itr'])]
    if verbose:
        print('last iteration only      ', len(MFX_Data))

    # Number of final localizations per trace
    unique_tids, inv_idx, locs_per_tid = np.unique(
        MFX_Data['tid'], return_inverse=True, return_counts=True
    )
    if verbose:
        print(f'min trace                 {np.min(locs_per_tid)}')

    # Keep only traces with at least 4 localizations
    MFX_Data = MFX_Data[locs_per_tid[inv_idx] > 3]
    if verbose:
        print('after trace length filt  ', len(MFX_Data))

    _, locs_per_tid_filt = np.unique(MFX_Data['tid'], return_counts=True)
    if verbose:
        print(f'Minimum trace             {np.min(locs_per_tid_filt)}')

    return MFX_Data

def plot(MFX_Data_1, MFX_Data_2, files, x1, y1, x2, y2, MFX_Data_3=None, x3=None, y3=None, name=None):
        
    # -----------------------------
    # Build Plotly figure with two traces (different colors)
    # -----------------------------
    fig = go.Figure()

    # File 1 - Orange
    fig.add_trace(
        go.Scattergl(
            x=x1, y=y1,
            mode="markers",
            name=os.path.basename(files[0]),
            marker=dict(
                size=3,
                opacity=0.9,
                color="orange",
            ),
            customdata=MFX_Data_1["tid"],
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                "tid=%{customdata}<br>"
                f"file={os.path.basename(files[0])}"
                "<extra></extra>"
            ),
        )
    )

    # File 2 - Red
    fig.add_trace(
        go.Scattergl(
            x=x2, y=y2,
            mode="markers",
            name=os.path.basename(files[1]),
            marker=dict(
                size=3,
                opacity=0.9,
                color="red",
            ),
            customdata=MFX_Data_2["tid"],
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                "tid=%{customdata}<br>"
                f"file={os.path.basename(files[1])}"
                "<extra></extra>"
            ),
        )
    )

    if MFX_Data_3 is not None and x3 is not None and y3 is not None:
        # File 3 - Blue
        fig.add_trace(
            go.Scattergl(
                x=x3, y=y3,
                mode="markers",
                name=name,
                marker=dict(
                    size=3,
                    opacity=0.9,
                    color="blue",
                ),
                customdata=MFX_Data_3["tid"],
                hovertemplate=(
                    "x=%{x:.3f}<br>"
                    "y=%{y:.3f}<br>"
                    "tid=%{customdata}<br>"
                    f"file={name}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title="MINFLUX localizations (2 channels merged)",
        template="plotly_white",
        dragmode="pan",
        margin=dict(l=10, r=10, t=40, b=10),
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)"
            ),
    )
    fig.layout.margin.autoexpand = False
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(uirevision="keep")
    fig.show()


In [5]:
# -----------------------------
# Load + process both MINFLUX data files
# -----------------------------
path = '/Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/for Max_MFX_echange_2colour_2D/'
files = [i for i in glob.glob(os.path.join(path, "**", "*.npy"), recursive=True) if i.endswith('.npy')]

print(f"Found {len(files)} files:")
for f in files:
    print(f"  - {os.path.basename(f)}")

# Process both files
MFX_Data_1 = filter_minflux_data(files[0])
MFX_Data_2 = filter_minflux_data(files[1])

# Convert locations to nanometers
locs_1 = MFX_Data_1['loc'] * 1e9
locs_2 = MFX_Data_2['loc'] * 1e9

x1, y1 = locs_1[:, 0], locs_1[:, 1]
x2, y2 = locs_2[:, 0], locs_2[:, 1]


Found 2 files:
  - 241008-133841_NUP96_minflux.npy
  - 241008-144515_NUP153_minflux.npy

--- Processing: 241008-133841_NUP96_minflux.npy ---
all raw                   89537
only valid_               89537
last iteration only       21237
min trace                 1
after trace length filt   21048
Minimum trace             4

--- Processing: 241008-144515_NUP153_minflux.npy ---
all raw                   13993
only valid_               13993
last iteration only       3259
min trace                 1
after trace length filt   3235
Minimum trace             4


In [6]:
plot(MFX_Data_1, MFX_Data_2, files, x1, y1, x2, y2)

In [ ]:
fileid = 1
grd_path = files[fileid].replace(os.path.basename(files[fileid]), 'grd/mbm/')
print(f"Trying to open Zarr file at: {grd_path}")

g = zarr.open_group(grd_path, mode="r")  # no need zarr_format=2 usually
arr = g['points']['xyz']

# Convert locations to nanometers
locs_3 = arr * 1e9

x3, y3 = locs_3[:, 0], locs_3[:, 1]

plot(MFX_Data_1, MFX_Data_2, files, x1, y1, x2, y2, MFX_Data_3=MFX_Data_2, x3=x3, y3=y3, name=os.path.basename(files[fileid]))


Trying to open Zarr file at: /Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/for Max_MFX_echange_2colour_2D/241008-144515_NUP153_minflux/grd/mbm/


## Overview MBM alignment 

A) Read MBM bead positions

From each dataset it loads:

    .../zarr/grd/mbm/points
    A table-like zarr array with fields like:
        xyz (Nx3 bead localizations over time)
        gri (integer bead id / group id)
    .../zarr/grd/mbm group attributes points_by_gri
    This is a mapping from gri → bead metadata (name, stickiness, times_failed_in_row, etc.)
    It marks beads as used=1 if times_failed_in_row < stickiness.

Then it computes for each bead an initial position (first element of a moving average of xyz) and keeps only beads:

    measured > 10 times
    present in both datasets
    z-outliers filtered (some logic using median/std in z)

So finally you have two matched point sets:

    ref_beads_pos (target)
    meas_beads_pos (source)


B) ICP (Iterative Closest Point)

Runs ICP from source to target, building a sequence of rigid transforms (rotation+translation) until mean displacement change < 0.5 nm.

C) Apply transform to MINFLUX localizations

Applies all the ICP transforms (in sequence) to the loc coordinates of the “moving” dataset, yielding aligned localizations.

### Plan

- Load MBM points table from grd/mbm/points for each dataset.
- For each bead (gri), compute one representative bead position (same as script: moving-average then take first).
- Match beads across datasets by gri (and/or name if you can access points_by_gri).
- Run ICP to compute transform(s).
- Apply transform(s) to channel 2 localizations.



In [2]:
import numpy as np
from scipy.spatial import cKDTree
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import webview
import os
import glob
import zarr

def filter_minflux_data(file_path, verbose=True):
    """Load and filter MINFLUX data from a single file."""
    MFX_Data = np.load(file_path)
    if verbose:
        print(f'\n--- Processing: {os.path.basename(file_path)} ---')
        print('all raw                  ', len(MFX_Data))

    # Keep only valid localizations
    MFX_Data = MFX_Data[MFX_Data['vld'] == True]
    if verbose:
        print('only valid_              ', len(MFX_Data))

    # Keep only last iteration
    MFX_Data = MFX_Data[MFX_Data['itr'] == np.max(MFX_Data['itr'])]
    if verbose:
        print('last iteration only      ', len(MFX_Data))

    # Number of final localizations per trace
    unique_tids, inv_idx, locs_per_tid = np.unique(
        MFX_Data['tid'], return_inverse=True, return_counts=True
    )
    if verbose:
        print(f'min trace                 {np.min(locs_per_tid)}')

    # Keep only traces with at least 4 localizations
    MFX_Data = MFX_Data[locs_per_tid[inv_idx] > 3]
    if verbose:
        print('after trace length filt  ', len(MFX_Data))

    _, locs_per_tid_filt = np.unique(MFX_Data['tid'], return_counts=True)
    if verbose:
        print(f'Minimum trace             {np.min(locs_per_tid_filt)}')

    return MFX_Data

def plot(MFX_Data_1, MFX_Data_2, files, x1, y1, x2, y2, MFX_Data_3=None, x3=None, y3=None, name=None):
        
    # -----------------------------
    # Build Plotly figure with two traces (different colors)
    # -----------------------------
    fig = go.Figure()

    # File 1 - Orange
    fig.add_trace(
        go.Scattergl(
            x=x1, y=y1,
            mode="markers",
            name=os.path.basename(files[0]),
            marker=dict(
                size=3,
                opacity=0.9,
                color="orange",
            ),
            customdata=MFX_Data_1["tid"],
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                "tid=%{customdata}<br>"
                f"file={os.path.basename(files[0])}"
                "<extra></extra>"
            ),
        )
    )

    # File 2 - Red
    fig.add_trace(
        go.Scattergl(
            x=x2, y=y2,
            mode="markers",
            name=os.path.basename(files[1]),
            marker=dict(
                size=3,
                opacity=0.9,
                color="red",
            ),
            customdata=MFX_Data_2["tid"],
            hovertemplate=(
                "x=%{x:.3f}<br>"
                "y=%{y:.3f}<br>"
                "tid=%{customdata}<br>"
                f"file={os.path.basename(files[1])}"
                "<extra></extra>"
            ),
        )
    )

    if MFX_Data_3 is not None and x3 is not None and y3 is not None:
        # File 3 - Blue
        fig.add_trace(
            go.Scattergl(
                x=x3, y=y3,
                mode="markers",
                name=name,
                marker=dict(
                    size=3,
                    opacity=0.9,
                    color="blue",
                ),
                customdata=MFX_Data_3["tid"],
                hovertemplate=(
                    "x=%{x:.3f}<br>"
                    "y=%{y:.3f}<br>"
                    "tid=%{customdata}<br>"
                    f"file={name}"
                    "<extra></extra>"
                ),
            )
        )

    fig.update_layout(
        title="MINFLUX localizations (2 channels merged)",
        template="plotly_white",
        dragmode="pan",
        margin=dict(l=10, r=10, t=40, b=10),
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)"
            ),
    )
    fig.layout.margin.autoexpand = False
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_layout(uirevision="keep")
    fig.show()


# 1. Utilities: moving average + rigid transform (Kabsch) + ICP

def moving_average_1d(a, n=4):
    a = np.asarray(a, dtype=float)
    if len(a) < n:
        return a
    ret = np.cumsum(a)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n-1:] / n

def best_rigid_transform(A, B):
    """
    Find R,t minimizing || (R A + t) - B || in least squares sense.
    A, B: (N,3)
    Returns R (3,3), t (3,)
    """
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B

    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # fix reflection
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = centroid_B - R @ centroid_A
    return R, t

def apply_T(points, T):
    """points: (N,3), T: (4,4)"""
    ones = np.ones((points.shape[0], 1))
    P = np.hstack([points, ones])
    out = (P @ T.T)[:, :3]
    return out

def icp(source, target, max_iterations=50, tolerance=0.5e-9):
    """
    source, target: (N,3) expected roughly corresponding.
    Returns:
      source_transformed, T_total, T_list
    """
    src = source.copy()
    T_total = np.eye(4)
    T_list = []
    prev_err = np.inf

    tree = cKDTree(target)

    for i in range(max_iterations):
        dists, idx = tree.query(src, k=1)
        corr = target[idx]

        R, t = best_rigid_transform(src, corr)

        T = np.eye(4)
        T[:3,:3] = R
        T[:3, 3] = t
        T_list.append(T)

        src = apply_T(src, T)

        # compose: new total is T * old when using row-vector convention in apply_T above
        T_total = T @ T_total

        mean_err = float(np.mean(dists))
        if abs(prev_err - mean_err) < tolerance:
            # converged
            break
        prev_err = mean_err

    return src, T_total, T_list

# 2. Load MBM beads from grd/mbm/points

def load_mbm_points(grd_mbm_path):
    """
    points is a zarr array (structured) with fields including xyz and gri

    grd_mbm_path should be the folder .../grd/mbm
    Returns structured array points (numpy) with fields e.g. 'xyz','gri',...
    """
    g = zarr.open_group(grd_mbm_path, mode="r")
    points = g["points"][:]   # read fully into memory
    return points, g

# 3. Compute one position per bead (replicates shared logic)

def bead_initial_positions(points, k=4, min_count=10):
    """
    - only beads with >10 measurements
    - moving average (k=4) on xyz
    - take the first averaged position as bead initial positon

    points: structured array with fields 'gri' and 'xyz' (N,3)
    Returns: dict {gri_int: pos(3,)}
    """
    gri_vals = np.asarray(points["gri"])
    xyz = np.asarray(points["xyz"])

    out = {}
    for gri in np.unique(gri_vals):
        mask = (gri_vals == gri)
        if mask.sum() <= min_count:
            continue
        bead_xyz = xyz[mask]

        x = moving_average_1d(bead_xyz[:,0], n=k)
        y = moving_average_1d(bead_xyz[:,1], n=k)
        z = moving_average_1d(bead_xyz[:,2], n=k)
        if len(x) == 0:
            continue

        out[int(gri)] = np.array([x[0], y[0], z[0]], dtype=float)
    return out

# 4. Match common beads + z-filter similar to shared code

def match_and_filter_beads(beads_ref, beads_mov):
    """
    beads_ref/mov: dict gri->(3,)
    Returns:
      ref_pts (N,3), mov_pts (N,3), common_gris (list)
    """
    common = sorted(set(beads_ref.keys()) & set(beads_mov.keys()))
    if len(common) < 3:
        raise ValueError(f"Need >=3 common beads for stable rigid transform, got {len(common)}")

    ref = np.vstack([beads_ref[g] for g in common])
    mov = np.vstack([beads_mov[g] for g in common])

    # replicate z outlier logic (in meters)
    z = ref[:,2]
    if np.std(z) > 100e-9:
        keep = (z < np.median(z) + 1.5*np.std(z))
    else:
        keep = (z < np.median(z) + 100e-9)

    common_kept = [g for g,k in zip(common, keep) if k]
    ref_kept = np.vstack([beads_ref[g] for g in common_kept])
    mov_kept = np.vstack([beads_mov[g] for g in common_kept])

    if len(common_kept) < 3:
        raise ValueError(f"After z-filter, need >=3 beads, got {len(common_kept)}")

    return ref_kept, mov_kept, common_kept

# 5. Apply to your MINFLUX loc array (channel 2 → channel 1)

def align_dataset_by_mbm(mfx_ref, mfx_mov, mbm_path_ref, mbm_path_mov, k=4):
    # load beads
    pts_ref, _ = load_mbm_points(mbm_path_ref)
    pts_mov, _ = load_mbm_points(mbm_path_mov)

    beads_ref = bead_initial_positions(pts_ref, k=k)
    beads_mov = bead_initial_positions(pts_mov, k=k)

    ref_pts, mov_pts, common = match_and_filter_beads(beads_ref, beads_mov)
    print(f"Using {len(common)} common beads (gri): {common[:10]}{'...' if len(common)>10 else ''}")

    # compute transform (move -> ref)
    mov_aligned_beads, T_total, T_list = icp(mov_pts, ref_pts, max_iterations=50, tolerance=0.5e-9)
    print("T_total:\n", T_total)

    # apply to MINFLUX localizations (only to loc, keep rest same)
    out = mfx_mov.copy()
    out_loc = apply_T(out["loc"], T_total)
    out["loc"] = out_loc
    return out, T_total

path = '/Users/maximiliansenftleben/Data/8376_alm/8376_minflux/data/for Max_MFX_echange_2colour_2D/'
files = [i for i in glob.glob(os.path.join(path, "**", "*.npy"), recursive=True) if i.endswith('.npy')]

# load and filter mfx
mfx1 = filter_minflux_data(files[0], verbose=False)
mfx2 = filter_minflux_data(files[1], verbose=False)

# locate mbm folders
mbm1 = os.path.join(os.path.dirname(files[0]), "grd", "mbm")
mbm2 = os.path.join(os.path.dirname(files[1]), "grd", "mbm")

mfx2_reg, T = align_dataset_by_mbm(mfx1, mfx2, mbm1, mbm2, k=4)

# plot before/after (nm)
plot(mfx1, mfx2, files,
     x1=(mfx1["loc"][:,0]*1e9), y1=(mfx1["loc"][:,1]*1e9),
     x2=(mfx2["loc"][:,0]*1e9), y2=(mfx2["loc"][:,1]*1e9))

plot(mfx1, mfx2, files,
     x1=(mfx1["loc"][:,0]*1e9), y1=(mfx1["loc"][:,1]*1e9),
     x2=(mfx2_reg["loc"][:,0]*1e9), y2=(mfx2_reg["loc"][:,1]*1e9),
     MFX_Data_3=None)

Using 7 common beads (gri): [686, 968, 1250, 1532, 1814, 2096, 2378]
T_total:
 [[ 9.99999706e-01  2.81840581e-04  7.12908013e-04 -2.14791543e-08]
 [-2.82535737e-04  9.99999485e-01  9.75186711e-04  5.55637707e-09]
 [-7.12632799e-04 -9.75387846e-04  9.99999270e-01  6.72042404e-10]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [39]:
# load and filter mfx
mfx1 = filter_minflux_data(files[0], verbose=False)
mfx2 = filter_minflux_data(files[1], verbose=False)

# locate mbm folders
mbm1 = os.path.join(os.path.dirname(files[0]), "grd", "mbm")
mbm2 = os.path.join(os.path.dirname(files[1]), "grd", "mbm")

mfx2_reg, T = align_dataset_by_mbm(mfx1, mfx2, mbm1, mbm2, k=4)

# plot before/after (nm)
plot(mfx1, mfx2, files,
     x1=(mfx1["loc"][:,0]*1e9), y1=(mfx1["loc"][:,1]*1e9),
     x2=(mfx2["loc"][:,0]*1e9), y2=(mfx2["loc"][:,1]*1e9))

plot(mfx1, mfx2, files,
     x1=(mfx1["loc"][:,0]*1e9), y1=(mfx1["loc"][:,1]*1e9),
     x2=(mfx2_reg["loc"][:,0]*1e9), y2=(mfx2_reg["loc"][:,1]*1e9),
     MFX_Data_3=None)

Using 7 common beads (gri): [686, 968, 1250, 1532, 1814, 2096, 2378]
T_total:
 [[ 9.99999706e-01  2.81840581e-04  7.12908013e-04 -2.14791543e-08]
 [-2.82535737e-04  9.99999485e-01  9.75186711e-04  5.55637707e-09]
 [-7.12632799e-04 -9.75387846e-04  9.99999270e-01  6.72042404e-10]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]
